# Exercise 4: Autoregressive Image Modeling
## Submission date: 25\05\2026, 23:59.

Submitted by:

## Configurations and initializations

This section loads libraries and configurations for various tasks for this course

In [ ]:
## Standard libraries
import os
import math
import numpy as np
import pandas as pd

## Imports for plotting
import matplotlib.pyplot as plt
plt.set_cmap('cividis')
%matplotlib inline
from IPython.display import set_matplotlib_formats
%config InlineBackend.figure_formats = ['svg', 'pdf']
from matplotlib.colors import to_rgb
import seaborn as sns

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim

# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms

# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = "../data"

# Path to the folder where the models are saved
CHECKPOINT_PATH = "./"

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

In [ ]:
# Convert images from 0-1 to 0-255 (integers).
def discretize(sample):
    return (sample * 255).to(torch.long)

# Transformations applied on each image => only make them a tensor
transform = transforms.Compose([transforms.ToTensor(), discretize])

# Loading the training dataset. We need to split it into a training and validation part
train_dataset = MNIST(root=DATASET_PATH, train=True, transform=transform, download=True)

#pl.seed_everything(42)
train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000])

# Loading the test set
test_set = MNIST(root=DATASET_PATH, train=False, transform=transform, download=True)

# We define a set of data loaders that we can use for various purposes later.
train_loader = data.DataLoader(train_set, batch_size=128, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = data.DataLoader(val_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)
test_loader = data.DataLoader(test_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)

print('Train size:', len(train_loader.dataset))
print('Validation size:', len(val_loader.dataset))
print('Test size:', len(test_loader.dataset))

In [ ]:
def show_imgs(imgs):
    num_imgs = imgs.shape[0] if isinstance(imgs, torch.Tensor) else len(imgs)
    nrow = min(num_imgs, 4)
    ncol = int(math.ceil(num_imgs/nrow))
    imgs = torchvision.utils.make_grid(imgs, nrow=nrow, pad_value=128)
    imgs = imgs.clamp(min=0, max=255)
    np_imgs = imgs.cpu().numpy()
    plt.figure(figsize=(1.5*nrow, 1.5*ncol))
    plt.imshow(np.transpose(np_imgs, (1,2,0)), interpolation='nearest')
    plt.axis('off')
    plt.show()

show_imgs([train_set[i][0] for i in range(8)])

## Define Masked Convolutions

In [ ]:
class MaskedConvolution(nn.Module):

    def __init__(self, c_in, c_out, mask, **kwargs):
        """
        Implements a convolution with mask applied on its weights.
        Inputs:
            c_in - Number of input channels
            c_out - Number of output channels
            mask - Tensor of shape [kernel_size_H, kernel_size_W] with 0s where
                   the convolution should be masked, and 1s otherwise.
            kwargs - Additional arguments for the convolution
        """
        super().__init__()
        # For simplicity: calculate padding automatically
        kernel_size = (mask.shape[0], mask.shape[1])

        # Actual convolution
        self.conv = nn.Conv2d(c_in, c_out, kernel_size, padding='same', **kwargs)

        # Mask as buffer => it is no parameter but still a tensor of the module
        # (must be moved with the devices)
        self.register_buffer('mask', mask[None,None].to(device))

    def forward(self, x):
        self.conv.weight.data *= self.mask # Ensures zero's at masked positions
        return self.conv(x)

In [ ]:
class VerticalStackConvolution(MaskedConvolution):

    def __init__(self, c_in, c_out, kernel_size=3, mask_center=False, **kwargs):
        # Mask out all pixels below. For efficiency, we could also reduce the kernel
        # size in height, but for simplicity, we stick with masking here.
        mask = torch.ones(kernel_size, kernel_size)
        mask[kernel_size//2+1:,:] = 0

        # For the very first convolution, we will also mask the center row
        if mask_center:
            mask[kernel_size//2,:] = 0

        super().__init__(c_in, c_out, mask, **kwargs)

class HorizontalStackConvolution(MaskedConvolution):

    def __init__(self, c_in, c_out, kernel_size=3, mask_center=False, **kwargs):
        # Mask out all pixels on the left. Note that our kernel has a size of 1
        # in height because we only look at the pixel in the same row.
        mask = torch.ones(1,kernel_size)
        mask[0,kernel_size//2+1:] = 0

        # For the very first convolution, we will also mask the center pixel
        if mask_center:
            mask[0,kernel_size//2] = 0

        super().__init__(c_in, c_out, mask, **kwargs)

## PixelCNN without Gating and dilations

Instead of the GatedMaskConv from the tutorial, we defined a MaskedConvLayer for the model.

In [ ]:
class MaskedConvLayer(nn.Module):
        def __init__(self, c_in, kernel_size=3, **kwargs):
            """
            Non Gated Convolution block implemented
            """
            super().__init__()
            self.kernel_size = kernel_size
            self.v_stack = VerticalStackConvolution(
                 c_in = c_in,
                 c_out = c_in,
                 kernel_size = self.kernel_size,
                 **kwargs
            )
            self.h_stack = HorizontalStackConvolution(
                 c_in = c_in,
                 c_out = c_in,
                 kernel_size = self.kernel_size,
                 **kwargs
            )

        def forward(self, v_stack, h_stack):
          v_stack_out = self.v_stack(v_stack)

          h_stack_out = self.h_stack(h_stack)
          h_stack_out = h_stack_out + v_stack_out

          v_stack_out = F.elu(v_stack_out)
          h_stack_out = F.elu(h_stack_out)
          
          return v_stack_out, h_stack_out

In [ ]:
class PixelCNN(nn.Module):

    def __init__(self, c_in, c_hidden, kernel_size=3, num_layers=5):
        super().__init__()

        self.kernel_size = kernel_size
        # First stacks/layers
        self.conv_vstack = VerticalStackConvolution(
                 c_in = c_in,
                 c_out = c_hidden,
                 kernel_size = self.kernel_size,
                 mask_center = True
            )
        self.conv_hstack = HorizontalStackConvolution(
                 c_in = c_in,
                 c_out = c_hidden,
                 kernel_size = self.kernel_size,
                 mask_center = True
            )
        
        # Rest of the layers, no mask
        self.conv_layers = nn.ModuleList(
            [MaskedConvLayer(c_in = c_hidden, kernel_size=self.kernel_size) for _ in range(num_layers)]
            )
        
        self.conv_out = nn.Conv2d(c_hidden, c_in * 256, kernel_size=1)
        

    def forward(self, x):
        """
        Forward image through model and return logits for each pixel.
        Inputs:
            x - Image tensor with integer values between 0 and 255.
        """
        # Scale input from 0 to 255 back to -1 to 1
        x = (x.float() / 255.0) * 2 - 1

        # Initial convolutions
        v_stack = self.conv_vstack(x)
        h_stack = self.conv_hstack(x)

        # Other Convolutions
        for layer in self.conv_layers:
            v_stack, h_stack = layer(v_stack, h_stack)

        # Apply ELU before 1x1 convolution for non-linearity on residual connection
        out = self.conv_out(F.elu(h_stack))

        # Output dimensions: [Batch, Classes, Channels, Height, Width]
        out = out.reshape(out.shape[0], 256, out.shape[1]//256, out.shape[2], out.shape[3])

        return out

    def calc_nll(self, x):
        # Forward pass with bpd (bits per dimension) negative log likelihood calculation
        out = self(x)
        loss = F.cross_entropy(out, x) / math.log(2)
        return loss

    @torch.no_grad()
    def sample(self, img_shape, img=None):
        """
        Sampling function for the autoregressive model.
        Inputs:
            img_shape - Shape of the image to generate (B,C,H,W)
            img (optional) - If given, this tensor will be used as
                             a starting image. The pixels to fill
                             should be -1 in the input tensor.
        """
        # Create empty image
        if img is None:
            img = torch.zeros(img_shape, dtype=torch.long).to(device) - 1
        # Generation loop
        for h in range(img_shape[2]):
            for w in range(img_shape[3]):
                for c in range(img_shape[1]):
                    # Skip if not to be filled (-1)
                    if (img[:,c,h,w] != -1).all().item():
                        continue
                    # For efficiency, we only have to input the upper part of the image
                    # as all other parts will be skipped by the masked convolutions anyways
                    pred = self.forward(img[:,:,:h+1,:])
                    probs = F.softmax(pred[:,:,c,h,w], dim=-1)
                    img[:,c,h,w] = torch.multinomial(probs, num_samples=1).squeeze(dim=-1)
        return img

Before you start training, you have to choose hyper-parameters for your model (e.g. number of layers).

We can check the full receptive field of the model on an MNIST image of size $28\times 28$ to help you decide which hyper-parameters to use:

In [ ]:
def show_center_recep_field(img, out):
    """
    Calculates the gradients of the input with respect to the output center pixel,
    and visualizes the overall receptive field.
    Inputs:
        img - Input image for which we want to calculate the receptive field on.
        out - Output features/loss which is used for backpropagation, and should be
              the output of the network/computation graph.
    """
    # Determine gradients
    loss = out[0,:,img.shape[2]//2,img.shape[3]//2].sum() # L1 loss for simplicity
    loss.backward(retain_graph=True) # Retain graph as we want to stack multiple layers and show the receptive field of all of them
    img_grads = img.grad.abs()
    img.grad.fill_(0) # Reset grads

    # Plot receptive field
    img = img_grads.squeeze().cpu().numpy()
    fig, ax = plt.subplots(1,2)
    pos = ax[0].imshow(img)
    ax[1].imshow(img>0)
    # Mark the center pixel in red if it doesn't have any gradients (should be the case for standard autoregressive models)
    show_center = (img[img.shape[0]//2,img.shape[1]//2] == 0)
    if show_center:
        center_pixel = np.zeros(img.shape + (4,))
        center_pixel[center_pixel.shape[0]//2,center_pixel.shape[1]//2,:] = np.array([1.0, 0.0, 0.0, 1.0])
    for i in range(2):
        ax[i].axis('off')
        if show_center:
            ax[i].imshow(center_pixel)
    ax[0].set_title("Weighted receptive field")
    ax[1].set_title("Binary receptive field")
    plt.show()
    plt.close()

In [ ]:
test_model = PixelCNN(c_in=1, c_hidden=64, num_layers=8).to('cpu')      # adjust according to your model
inp = torch.zeros(1,1,28,28).to('cpu')                    # do not remove the 'cpu' here
inp.requires_grad_()
out = test_model(inp)
show_center_recep_field(inp, out.squeeze(dim=2))
del inp, out, test_model

## Model Training

In [ ]:
model = PixelCNN(c_in=1, c_hidden=128, kernel_size=3)       # adjust according to your model
print("Num params: {:,}".format(sum(p.numel() for p in model.parameters())))

model = torch.compile(model.to(device))

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.99)
epochs = 500

In [ ]:
# Training loop
train_bpd = []
val_bpd = []

for epoch in range(epochs):
    model.train()
    losses = []

    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.calc_nll(imgs)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    train_bpd.append(np.mean(losses))
    scheduler.step()

    model.eval()
    losses = []
    with torch.no_grad():
      for imgs, _ in val_loader:
          imgs = imgs.to(device)
          loss = model.calc_nll(imgs)
          losses.append(loss.item())

    val_bpd.append(np.mean(losses))

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_bpd[-1]:.4f}, Val Loss: {val_bpd[-1]:.4f}")

    if (epoch + 1) % 5 == 0:
       sampled_images = model.sample(img_shape=(4, 1, 28, 28))
       show_imgs(sampled_images)

# Test loop
model.eval()
losses = []
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        loss = model.calc_nll(imgs)
        losses.append(loss.item())

test_bpd = np.mean(losses)
print(f"Test NLL: {test_bpd:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), train_bpd, label='Train NLL (bpd)', color='blue')
plt.plot(range(1, epochs + 1), val_bpd, label='Validation NLL (bpd)', color='orange')
plt.xlabel('Epochs')
plt.ylabel('Negative Log-Likelihood (bpd)')
plt.title('Training and Validation NLL Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

## Question 1

## Question 2

## Question 3